# STEP 9 - adversarial training, the defense a reviewer will propose instead

"Why not just put the attack in the training data?" is the first alternative any reviewer
will suggest, and the paper currently has no answer. This notebook gets one.

We train **two XLM-R models in the same notebook**, identical except for the training data:

- **baseline** - the normal training set.
- **adversarially trained** - half the training comments additionally appear with the
  rendering-verified ZWNJ attack applied.

Then we attack both with four things: the ZWNJ attack it was trained on, the **same attack
using ZWJ instead**, the **hasant-adjacent** variant, and **ZWSP**. The question is not
whether adversarial training helps on the attack it saw. It is whether it generalises to
invisible characters it did not see, and what it costs.

Same protocol as Step 6 so the baseline is comparable: full training set, batch 32,
2 epochs, lr 2e-5, fp16, seed 42.

**Settings:** GPU **T4 x2**, Internet **ON**, data **`naurosromim/bdshs`**. About 1 hour.


In [ ]:
!pip install -q "emoji==1.7.0" regex ftfy uharfbuzz
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
%%writefile /kaggle/working/render_attack.py
"""
Rendering-verified zero-width attacks and defenses for Bangla.

Threat model
------------
An insertion counts as imperceptible only if HarfBuzz shapes the word to the
same glyph ids and glyph positions, in every reference font, with and without
it. HarfBuzz is the shaper used by Chrome, Android and Firefox.

The earlier rule-based attacks (strict / semi) are kept for comparison, but
every number in the paper should come from the render-gated functions here.
"""
import os
import re
import random
import urllib.request
import unicodedata
from functools import lru_cache

import uharfbuzz as hb

ZWNJ, ZWJ, ZWSP = "‌", "‍", "​"
HASANT = "্"
ZW = (ZWNJ, ZWJ, ZWSP)

# Open-licensed Bangla fonts, all downloadable on Kaggle.
FONT_URLS = {
    "NotoSansBengali":  "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSansBengali/unhinted/ttf/NotoSansBengali-Regular.ttf",
    "NotoSerifBengali": "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSerifBengali/unhinted/ttf/NotoSerifBengali-Regular.ttf",
    "HindSiliguri":     "https://raw.githubusercontent.com/google/fonts/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
    "TiroBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/tirobangla/TiroBangla-Regular.ttf",
    "BalooDa2":         "https://raw.githubusercontent.com/google/fonts/main/ofl/balooda2/BalooDa2%5Bwght%5D.ttf",
    "AnekBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/anekbangla/AnekBangla%5Bwdth,wght%5D.ttf",
    "Galada":           "https://raw.githubusercontent.com/google/fonts/main/ofl/galada/Galada-Regular.ttf",
    "Mina":             "https://raw.githubusercontent.com/google/fonts/main/ofl/mina/Mina-Regular.ttf",
}


def fetch_fonts(dest, names=None):
    os.makedirs(dest, exist_ok=True)
    paths = {}
    for name in names or FONT_URLS:
        p = os.path.join(dest, name + ".ttf")
        if not os.path.exists(p):
            urllib.request.urlretrieve(FONT_URLS[name], p)
        paths[name] = p
    return paths


def is_bangla(ch):
    """True for one Bangla code point.

    An empty string returns False, so callers may pass a missing neighbour
    (start or end of string) without guarding first."""
    return len(ch) == 1 and 0x0980 <= ord(ch) <= 0x09FF


# ------------------------------------------------------------------ slots

def inert_slots(word):
    """Between two Bangla chars, neither of them hasant (the old `semi` set)."""
    return [i for i in range(1, len(word))
            if is_bangla(word[i - 1]) and is_bangla(word[i])
            and HASANT not in (word[i - 1], word[i])]


def strict_slots(word):
    """Both neighbours Bangla base letters (the old `strict` set)."""
    return [i for i in inert_slots(word)
            if unicodedata.category(word[i - 1]) == "Lo"
            and unicodedata.category(word[i]) == "Lo"]


def hasant_slots(word):
    """Immediately before or after a hasant. Position-aware defense keeps these."""
    out = set()
    for h, c in enumerate(word):
        if c != HASANT:
            continue
        if h > 0 and is_bangla(word[h - 1]):
            out.add(h)
        if h + 1 < len(word) and is_bangla(word[h + 1]):
            out.add(h + 1)
    return sorted(out)


SLOT_MODES = {
    "inert":  inert_slots,
    "strict": strict_slots,
    "hasant": hasant_slots,
    "both":   lambda w: inert_slots(w) + hasant_slots(w),
}


def _build(word, chosen, char):
    return "".join((char if i in chosen else "") + c for i, c in enumerate(word))


def _parts(text):
    return re.split(r"(\s+)", text)


# --------------------------------------------------------------- renderer

class Renderer:
    def __init__(self, font_paths):
        self.fonts = {n: hb.Font(hb.Face(hb.Blob.from_file_path(p), 0))
                      for n, p in font_paths.items()}
        self.names = list(self.fonts)
        self._shape = lru_cache(maxsize=2_000_000)(self._shape_uncached)

    def _shape_uncached(self, name, text):
        buf = hb.Buffer()
        buf.add_str(text)
        buf.guess_segment_properties()
        buf.flags = hb.BufferFlags.REMOVE_DEFAULT_IGNORABLES
        hb.shape(self.fonts[name], buf, {})
        return (tuple(g.codepoint for g in buf.glyph_infos),
                tuple((p.x_advance, p.y_advance, p.x_offset, p.y_offset)
                      for p in buf.glyph_positions))

    def same(self, a, b, fonts=None):
        return all(self._shape(n, a) == self._shape(n, b) for n in (fonts or self.names))

    def visible_in(self, a, b):
        """Names of fonts in which a and b render differently."""
        return [n for n in self.names if self._shape(n, a) != self._shape(n, b)]

    # ------------------------------------------------------------ attacks

    def attack(self, text, budget="ALL", char=ZWNJ, seed=0, mode="inert", fonts=None):
        """Insert zero-width chars only where the word still renders identically.

        Candidates are shuffled and accepted one at a time, each checked against
        the clean word with all previously accepted insertions in place.
        Returns (attacked_text, n_inserted)."""
        parts = _parts(text)
        rng = random.Random(seed)
        slots = SLOT_MODES[mode]
        cands = [(pi, i) for pi, p in enumerate(parts)
                 if p and not p.isspace() for i in slots(p)]
        rng.shuffle(cands)
        chosen, n = {}, 0
        for pi, i in cands:
            if budget != "ALL" and n >= budget:
                break
            trial = chosen.get(pi, set()) | {i}
            if self.same(parts[pi], _build(parts[pi], trial, char), fonts):
                chosen[pi] = trial
                n += 1
        out = "".join(_build(p, chosen[pi], char) if pi in chosen else p
                      for pi, p in enumerate(parts))
        return out, n

    # ------------------------------------------------------------ defense

    def defend_render_equivalence(self, text, fonts=None):
        """Delete a zero-width char iff removing it leaves rendering unchanged in
        every reference font. Only chars whose nearest non-zero-width neighbours
        on both sides are Bangla are considered, so emoji ZWJ sequences survive."""
        parts = _parts(text)
        for pi, w in enumerate(parts):
            if not any(z in w for z in ZW):
                continue
            i = 0
            while i < len(w):
                if w[i] in ZW:
                    l = i - 1
                    while l >= 0 and w[l] in ZW:
                        l -= 1
                    r = i + 1
                    while r < len(w) and w[r] in ZW:
                        r += 1
                    if l >= 0 and r < len(w) and is_bangla(w[l]) and is_bangla(w[r]):
                        cand = w[:i] + w[i + 1:]
                        if self.same(w, cand, fonts):
                            w = cand
                            continue
                i += 1
            parts[pi] = w
        return "".join(parts)


# ------------------------------------------------ rule-based (for comparison)

def attack_ungated(text, budget="ALL", char=ZWNJ, seed=0, mode="inert"):
    parts = _parts(text)
    rng = random.Random(seed)
    slots = SLOT_MODES[mode]
    cands = [(pi, i) for pi, p in enumerate(parts) if p and not p.isspace() for i in slots(p)]
    rng.shuffle(cands)
    if budget != "ALL":
        cands = cands[:budget]
    chosen = {}
    for pi, i in cands:
        chosen.setdefault(pi, set()).add(i)
    out = "".join(_build(p, chosen[pi], char) if pi in chosen else p for pi, p in enumerate(parts))
    return out, len(cands)


def defend_strip_all(text):
    for z in ZW:
        text = text.replace(z, "")
    return text


def defend_nfc(text):
    return unicodedata.normalize("NFC", text)


def defend_position_aware(text):
    out = []
    for i, ch in enumerate(text):
        if ch in (ZWNJ, ZWJ):
            prev = text[i - 1] if i > 0 else ""
            nxt = text[i + 1] if i + 1 < len(text) else ""
            if HASANT in (prev, nxt):
                out.append(ch)
            continue
        if ch == ZWSP:
            continue
        out.append(ch)
    return "".join(out)


def zw_stats(texts):
    docs = total = functional = 0
    for t in texts:
        hit = False
        for i, ch in enumerate(t):
            if ch not in (ZWNJ, ZWJ):
                continue
            hit = True
            total += 1
            prev = t[i - 1] if i > 0 else ""
            nxt = t[i + 1] if i + 1 < len(t) else ""
            functional += HASANT in (prev, nxt)
        docs += hit
    return dict(n_docs=len(texts), docs_with_zw=docs, total_zw=total,
                functional=functional, inert=total - functional)


In [ ]:
import sys, os, glob, time, random, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/kaggle/working")
import importlib, render_attack as ra; importlib.reload(ra)
import numpy as np, pandas as pd, torch
from functools import lru_cache
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
from normalizer import normalize

DEV, OUT = "cuda", "/kaggle/working"
R = ra.Renderer(ra.fetch_fonts(f"{OUT}/fonts"))
paths = glob.glob("/kaggle/input/**/*.csv", recursive=True)
pick = lambda k: [p for p in paths if k in os.path.basename(p).lower()][0]
rtr, rte = pd.read_csv(pick("train")), pd.read_csv(pick("test"))
obj = [c for c in rtr.columns if rtr[c].dtype == object]
TEXT = max(obj, key=lambda c: rtr[c].astype(str).str.len().mean())
LABEL = [c for c in rtr.columns if c != TEXT and rtr[c].dropna().nunique() == 2][0]
def prep(d):
    d = d[[TEXT, LABEL]].dropna().copy(); d.columns = ["text", "label"]
    d["text"] = d["text"].astype(str); d["label"] = d["label"].astype(int)
    return d[d.text.str.strip().str.len() > 0].reset_index(drop=True)
train, test = prep(rtr), prep(rte)
HATE = 1
print("train", len(train), "| test", len(test))

# --- training sets: identical except for the augmentation
rng = random.Random(0)
aug_idx = rng.sample(range(len(train)), len(train) // 2)
t0 = time.time()
aug_txt = [R.attack(train.text[i], "ALL", mode="inert", seed=9000 + i)[0] for i in aug_idx]
print(f"built {len(aug_txt)} attacked training copies in {time.time()-t0:.0f}s")
TRAIN_SETS = {
    "baseline":    (train.text.tolist(), train.label.tolist()),
    "adv_trained": (train.text.tolist() + aug_txt,
                    train.label.tolist() + [int(train.label[i]) for i in aug_idx]),
}
for k, (x, y) in TRAIN_SETS.items(): print(f"  {k:12} {len(x)} examples")

In [ ]:
# --- four attacks. Only the first is what adversarial training saw.
N_ATK = 1500
r2 = random.Random(0)
hate_txt = [test.text[i] for i in sorted(r2.sample(test.index[test.label == HATE].tolist(), N_ATK))]
ben_txt  = [test.text[i] for i in sorted(r2.sample(test.index[test.label != HATE].tolist(), N_ATK))]
ATK = {
    "ZWNJ (seen in training)":  [R.attack(t, "ALL", mode="inert", char=ra.ZWNJ, seed=i)[0] for i, t in enumerate(hate_txt)],
    "ZWJ (unseen character)":   [R.attack(t, "ALL", mode="inert", char=ra.ZWJ,  seed=i)[0] for i, t in enumerate(hate_txt)],
    # ZWSP is deleted by the csebuetnlp normalizer before tokenization, so this row
    # must show 0% for BOTH models. It is a pipeline control, not evidence that
    # adversarial training generalises. Labelled so it cannot be misread.
    "ZWSP (normalizer removes it)": [R.attack(t, "ALL", mode="inert", char=ra.ZWSP, seed=i)[0] for i, t in enumerate(hate_txt)],
    "hasant (unseen position)": [R.attack(t, "ALL", mode="hasant", char=ra.ZWNJ, seed=i)[0] for i, t in enumerate(hate_txt)],
}
for k, v in ATK.items():
    vis = np.mean([len(R.visible_in(c, a)) > 0 for c, a in zip(hate_txt, v)])
    print(f"  {k:26} visible in any font: {vis:.4f}")
# our defense needs no retraining at all
DEFENDED = {k: [R.defend_render_equivalence(t) for t in v] for k, v in ATK.items()}

In [ ]:
@lru_cache(maxsize=None)
def norm(t): return normalize(t)
MAXLEN, BATCH, EPOCHS, LR, SEED = 128, 32, 2, 2e-5, 42

def run(tag, texts, labels):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    tk = AutoTokenizer.from_pretrained("xlm-roberta-base")
    mdl = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=2).to(DEV)
    e = tk([norm(t) for t in texts], truncation=True, padding="max_length",
           max_length=MAXLEN, return_tensors="pt")
    y = torch.tensor(labels)
    g = torch.Generator(); g.manual_seed(SEED)
    dl = DataLoader(TensorDataset(e["input_ids"], e["attention_mask"], y),
                    batch_size=BATCH, shuffle=True, generator=g)
    opt = torch.optim.AdamW(mdl.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS*len(dl), pct_start=0.1)
    sc = torch.cuda.amp.GradScaler(); mdl.train()
    for ep in range(EPOCHS):
        for st, (i_, m_, l_) in enumerate(dl):
            i_, m_, l_ = i_.to(DEV), m_.to(DEV), l_.to(DEV); opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = mdl(input_ids=i_, attention_mask=m_, labels=l_).loss
            sc.scale(loss).backward(); sc.step(opt); sc.update(); sch.step()
            if st % 500 == 0: print(f"    {tag} ep{ep} {st}/{len(dl)} loss {loss.item():.4f}")
    mdl.eval()

    @torch.no_grad()
    def pred(ts, bs=128):
        ids = tk([norm(t) for t in ts], truncation=True, max_length=512)["input_ids"]
        order = np.argsort([len(z) for z in ids]); out = np.empty(len(ts), dtype=int)
        for s in range(0, len(order), bs):
            idx = order[s:s+bs]
            b = tk.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt")
            with torch.cuda.amp.autocast():
                lg = mdl(input_ids=b["input_ids"].to(DEV), attention_mask=b["attention_mask"].to(DEV)).logits
            out[idx] = lg.float().argmax(-1).cpu().numpy()
        return out

    ty = test.label.values; cp = pred(test.text.tolist())
    acc, f1 = accuracy_score(ty, cp), f1_score(ty, cp)
    hp = pred(hate_txt); tgt = np.where(hp == HATE)[0]
    rows = []
    for k in ATK:
        ev = float((pred([ATK[k][i] for i in tgt]) != HATE).mean())
        evd = float((pred([DEFENDED[k][i] for i in tgt]) != HATE).mean())
        rows.append({"model": tag, "attack": k, "n_targets": len(tgt),
                     "evasion": round(ev, 4), "evasion_with_our_defense": round(evd, 4)})
        print("   ", rows[-1])
    bp, bpa = pred(ben_txt), pred([ATK["ZWNJ (seen in training)"][i] for i in range(len(ben_txt))])
    clean = {"model": tag, "clean_acc": round(acc, 4), "clean_f1": round(f1, 4),
             "benign_fpr_clean": round(float((bp == HATE).mean()), 4),
             "benign_fpr_attacked": round(float((bpa == HATE).mean()), 4)}
    print("   ", clean)
    del mdl; torch.cuda.empty_cache()
    return rows, clean

In [ ]:
EV, CL = [], []
for tag, (x, y) in TRAIN_SETS.items():
    t0 = time.time()
    r, c = run(tag, x, y)
    EV += r; CL.append(c)
    pd.DataFrame(EV).to_csv(f"{OUT}/step9_advtrain_evasion.csv", index=False)
    pd.DataFrame(CL).to_csv(f"{OUT}/step9_advtrain_clean.csv", index=False)
    print(f"== {tag} done in {(time.time()-t0)/60:.1f} min\n")

In [ ]:
pd.set_option("display.width", 250)
ev = pd.DataFrame(EV); cl = pd.DataFrame(CL)
print("="*92); print("STEP 9 SUMMARY - adversarial training vs our defense"); print("="*92)
print("\nNOTE: the ZWSP row is a pipeline control. The normalizer deletes ZWSP, so 0%")
print("      there says nothing about adversarial training.")
print(ev.pivot(index="attack", columns="model", values="evasion").to_string())
print("\n-- with render-equivalence applied on top (no retraining) --")
print(ev.pivot(index="attack", columns="model", values="evasion_with_our_defense").to_string())
print("\n-- what adversarial training cost --"); print(cl.to_string(index=False))
print("="*92)